# 15 — Callable Bonds

A callable bond gives the issuer the right to redeem early.

- **Black model** pricing
- **Hull-White trinomial-tree** pricing
- QL comparison
- OAS (option-adjusted spread) via AD
- Call-schedule sensitivity

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
from ql_jax.instruments.callable_bond import (
    CallScheduleEntry, CallableFixedRateBond, black_callable_bond_price)
from ql_jax.engines.bond.callable import CallableBond, callable_bond_price_hw

# 10-year bond, callable from year 3 onward
FACE = 100.0
COUPON = 0.05
MATURITY = 10.0
FLAT_RATE = 0.05
YIELD_VOL = 0.10
HW_A = 0.1
HW_SIGMA = 0.01

coupon_times = jnp.array([float(i) for i in range(1, 11)])
call_schedule = [CallScheduleEntry(exercise_time=float(t), price=100.0)
                 for t in range(3, 10)]

---
## 1. Black Model

In [ ]:
bond_black = CallableFixedRateBond(
    face=FACE, coupon_rate=COUPON,
    coupon_times=coupon_times, maturity=MATURITY,
    call_schedule=call_schedule)

dc_t = jnp.linspace(0.01, 11, 50)
dc_v = jnp.exp(-FLAT_RATE * dc_t)

p_black = float(black_callable_bond_price(bond_black, YIELD_VOL, dc_t, dc_v))
print(f"Callable bond (Black, 10% yield vol) : {p_black:.6f}")

---
## 2. Hull-White Tree

In [ ]:
bond_hw = CallableBond(
    face_value=FACE, coupon_rate=COUPON,
    coupon_dates=coupon_times,
    call_dates=jnp.array([float(t) for t in range(3, 10)]),
    call_prices=jnp.full(7, 100.0),
    maturity=MATURITY)

disc_fn = lambda t: jnp.exp(-FLAT_RATE * t)

p_hw = float(callable_bond_price_hw(bond_hw, HW_A, HW_SIGMA, disc_fn, n_steps=100))
print(f"Callable bond (HW tree, a={HW_A}, σ={HW_SIGMA}) : {p_hw:.6f}")

---
## 3. Straight Bond Comparison

In [ ]:
# Straight bond = sum of discounted cashflows
cfs = jnp.array([FACE * COUPON] * 10)
cfs = cfs.at[-1].add(FACE)
p_straight = float(jnp.sum(cfs * jnp.exp(-FLAT_RATE * coupon_times)))

option_value_black = p_straight - p_black
option_value_hw    = p_straight - p_hw

print(f"Straight bond        : {p_straight:.6f}")
print(f"Callable (Black)     : {p_black:.6f}  → call option value = {option_value_black:.4f}")
print(f"Callable (HW tree)   : {p_hw:.6f}  → call option value = {option_value_hw:.4f}")

---
## 4. Yield Vol Sensitivity

In [ ]:
import matplotlib.pyplot as plt

vols = np.linspace(0.01, 0.30, 30)
black_prices = [float(black_callable_bond_price(bond_black, v, dc_t, dc_v)) for v in vols]

plt.figure(figsize=(8, 5))
plt.plot(vols * 100, black_prices, 'b-', linewidth=2)
plt.axhline(p_straight, color='gray', ls='--', label='Straight bond')
plt.xlabel('Yield Vol (%)')
plt.ylabel('Callable Bond Price')
plt.title('Callable Bond Price vs Yield Volatility')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 5. AD: Duration via Automatic Differentiation

In [ ]:
def callable_price_rate(rate):
    dc_v_r = jnp.exp(-rate * dc_t)
    return black_callable_bond_price(bond_black, YIELD_VOL, dc_t, dc_v_r)

dp_dr = float(jax.grad(callable_price_rate)(FLAT_RATE))
mod_dur = -dp_dr / p_black

# Convexity via second derivative
d2p_dr2 = float(jax.grad(jax.grad(callable_price_rate))(FLAT_RATE))
convexity = d2p_dr2 / p_black

print(f"dP/dr        : {dp_dr:.4f}")
print(f"Modified dur : {mod_dur:.4f}")
print(f"Convexity    : {convexity:.4f}")

---
## 6. Exercises

1. **Puttable bond**: Create a bond with put schedule and compare the option component.
2. **HW convergence**: Plot HW tree price vs `n_steps` to verify convergence.
3. **OAS**: Given a market price, find the OAS by root-finding with `jax.grad`.

---
*End of Notebook 15*